In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = (SparkSession.builder
         .appName("windows")
         .master("spark://spark-master:7077")
         .config("spark.executor.memory", "512m")
         .getOrCreate())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/27 11:30:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = (spark.read.format("json")
      .option("multiLine", "true")
      .load("../data/nobel_prizes.json")
     )

df.show()

+----------+--------------------+--------------------+----+
|  category|           laureates|   overallMotivation|year|
+----------+--------------------+--------------------+----+
| chemistry|[{Carolyn, 1015, ...|                null|2022|
| economics|[{Ben, 1021, "for...|                null|2022|
|literature|[{Annie, 1017, "f...|                null|2022|
|     peace|[{Ales, 1018, "Th...|                null|2022|
|   physics|[{Alain, 1012, "f...|                null|2022|
|  medicine|[{Svante, 1011, "...|                null|2022|
| chemistry|[{Benjamin, 1002,...|                null|2021|
| economics|[{David, 1007, "f...|                null|2021|
|literature|[{Abdulrazak, 100...|                null|2021|
|     peace|[{Maria, 1005, "f...|                null|2021|
|   physics|[{Syukuro, 999, "...|"for groundbreaki...|2021|
|  medicine|[{David, 997, "fo...|                null|2021|
| chemistry|[{Emmanuelle, 991...|                null|2020|
| economics|[{Paul, 995, "for...|       

In [4]:
df_flattened = (
    df
    .withColumn("laureates",
                explode(col("laureates")))
    .select(col("category")
            ,col("year")
            ,col("overallMotivation")
            ,col("laureates.id")
            ,col("laureates.firstname")
            ,col("laureates.surname")
            ,col("laureates.share")
            ,col("laureates.motivation")))

df_flattened.show()

+----------+----+-----------------+----+--------------------+-----------+-----+--------------------+
|  category|year|overallMotivation|  id|           firstname|    surname|share|          motivation|
+----------+----+-----------------+----+--------------------+-----------+-----+--------------------+
| chemistry|2022|             null|1015|             Carolyn|   Bertozzi|    3|"for the developm...|
| chemistry|2022|             null|1016|              Morten|     Meldal|    3|"for the developm...|
| chemistry|2022|             null| 743|               Barry|  Sharpless|    3|"for the developm...|
| economics|2022|             null|1021|                 Ben|   Bernanke|    3|"for research on ...|
| economics|2022|             null|1022|             Douglas|    Diamond|    3|"for research on ...|
| economics|2022|             null|1023|              Philip|     Dybvig|    3|"for research on ...|
|literature|2022|             null|1017|               Annie|     Ernaux|    1|"for the cou

In [5]:
df_dropna = df_flattened.dropna()

df_dropna.show()

+---------+----+--------------------+----+----------+----------+-----+--------------------+
| category|year|   overallMotivation|  id| firstname|   surname|share|          motivation|
+---------+----+--------------------+----+----------+----------+-----+--------------------+
|  physics|2021|"for groundbreaki...| 999|   Syukuro|    Manabe|    4|"for the physical...|
|  physics|2021|"for groundbreaki...|1000|     Klaus|Hasselmann|    4|"for the physical...|
|  physics|2021|"for groundbreaki...|1001|   Giorgio|    Parisi|    2|"for the discover...|
|  physics|2019|"for contribution...| 973|     James|   Peebles|    2|"for theoretical ...|
|  physics|2019|"for contribution...| 974|    Michel|     Mayor|    4|"for the discover...|
|  physics|2019|"for contribution...| 975|    Didier|    Queloz|    4|"for the discover...|
|  physics|2018|"for groundbreaki...| 960|    Arthur|    Ashkin|    2|"for the optical ...|
|  physics|2018|"for groundbreaki...| 961|    Gérard|    Mourou|    4|"for their

In [7]:
df_fillna = df_flattened.fillna("N/A")

df_fillna.show()

+----------+----+-----------------+----+--------------------+-----------+-----+--------------------+
|  category|year|overallMotivation|  id|           firstname|    surname|share|          motivation|
+----------+----+-----------------+----+--------------------+-----------+-----+--------------------+
| chemistry|2022|              N/A|1015|             Carolyn|   Bertozzi|    3|"for the developm...|
| chemistry|2022|              N/A|1016|              Morten|     Meldal|    3|"for the developm...|
| chemistry|2022|              N/A| 743|               Barry|  Sharpless|    3|"for the developm...|
| economics|2022|              N/A|1021|                 Ben|   Bernanke|    3|"for research on ...|
| economics|2022|              N/A|1022|             Douglas|    Diamond|    3|"for research on ...|
| economics|2022|              N/A|1023|              Philip|     Dybvig|    3|"for research on ...|
|literature|2022|              N/A|1017|               Annie|     Ernaux|    1|"for the cou

In [8]:
df_replace = (
    df_flattened
    .withColumn("category", when(col("category").isNull(), "").otherwise(col("category")))
    .withColumn("overallMotivation", when(col("overallMotivation").isNull(), "").otherwise(col("overallMotivation")))
    .withColumn("firstname", when(col("firstname").isNull(), "").otherwise(col("firstname")))
    .withColumn("surname", when(col("surname").isNull(), "").otherwise(col("surname")))
    .withColumn("year", when(col("year").isNull(), 9999).otherwise(col("year"))))

df_replace.show()

+----------+----+-----------------+----+--------------------+-----------+-----+--------------------+
|  category|year|overallMotivation|  id|           firstname|    surname|share|          motivation|
+----------+----+-----------------+----+--------------------+-----------+-----+--------------------+
| chemistry|2022|                 |1015|             Carolyn|   Bertozzi|    3|"for the developm...|
| chemistry|2022|                 |1016|              Morten|     Meldal|    3|"for the developm...|
| chemistry|2022|                 | 743|               Barry|  Sharpless|    3|"for the developm...|
| economics|2022|                 |1021|                 Ben|   Bernanke|    3|"for research on ...|
| economics|2022|                 |1022|             Douglas|    Diamond|    3|"for research on ...|
| economics|2022|                 |1023|              Philip|     Dybvig|    3|"for research on ...|
|literature|2022|                 |1017|               Annie|     Ernaux|    1|"for the cou

In [9]:
from pyspark.ml.feature import *

data = [
    (1, 2.0),
    (2, None),
    (3, 5.0),
    (4, None),
    (5, 7.0)
]

df = spark.createDataFrame(data, ["id", "value"])

imputer = Imputer(inputCols=["value"], outputCols=["imputed_value"])

imputer_model = imputer.fit(df)
imputed_df = imputer_model.transform(df)

imputed_df.show()

+---+-----+-----------------+
| id|value|    imputed_value|
+---+-----+-----------------+
|  1|  2.0|              2.0|
|  2| null|4.666666666666667|
|  3|  5.0|              5.0|
|  4| null|4.666666666666667|
|  5|  7.0|              7.0|
+---+-----+-----------------+



In [10]:
spark.stop()